### Lading - Preparação e inspeção

In [0]:
import re
from datetime import datetime

# Padronização dos nomes dos schemas e volumes
CATALOG = "vcredit_medallion"
BRONZE_SCHEMA_NAME = "bronze"
SILVER_SCHEMA_NAME = "silver"
GOLD_SCHEMA_NAME = "gold"

BRONZE_SCHEMA = f"{CATALOG}.{BRONZE_SCHEMA_NAME}"
SILVER_SCHEMA = f"{CATALOG}.{SILVER_SCHEMA_NAME}"
GOLD_SCHEMA = f"{CATALOG}.{GOLD_SCHEMA_NAME}"

LANDING_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA_NAME}/landing"

# Símbolos padronizados de log
OK = "✅"
FAIL = "❌"
WARN = "⚠️"

# Para verificação visual
print(f"CATALOG: {CATALOG}")
print(f"BRONZE_SCHEMA: {BRONZE_SCHEMA}")
print(f"SILVER_SCHEMA: {SILVER_SCHEMA}")
print(f"GOLD_SCHEMA: {GOLD_SCHEMA}")
print(f"LANDING_PATH: {LANDING_PATH}")



In [0]:
# Cria, apenas se já não existem os schemas e volumes garantindo idempotência.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {BRONZE_SCHEMA}.landing")

In [0]:
# Utiliza o módulo re (regex) para corrigir possíveis títulos errados antes de verificar a presença do arquivo.
# Cria o dicionário contento os nomes dos arquivos esperados bem como os respectivos regex para buscar nomes diferentes (contendo ano, por exemplo)
# e não diferenciando maiúsculas e minúsculas.
expected_files = {
    "base_atendentes.csv": r"(?i)base_atendentes.*\.csv$",
    "base_motivos.csv": r"(?i)base_motivos.*\.csv$",
    "canais.csv": r"(?i)canais.*\.csv$",
    "chamados.csv": r"(?i)chamados.*\.csv$",
    "chamados_hora.csv": r"(?i)Chamados_Hora.*\.csv$",
    "clientes.csv": r"(?i)clientes.*\.csv$",
    "custos.csv": r"(?i)custos.*\.csv$",
    "pesquisa_satisfacao.csv": r"(?i)pesquisa_satisfacao.*\.csv$",
}

# Verifica se todos os arquivos estão presentes e corrige o nome se necessário, removendo datas, por exemplo.
file_infos = dbutils.fs.ls(LANDING_PATH)
found_files = [f.name for f in file_infos]

empty_files = [f.name for f in file_infos if f.size == 0]
if empty_files:
    print(f"{FAIL} Arquivos vazios detectados: {empty_files}")
    raise Exception(f"Arquivos vazios encontrados: {empty_files}")

print("Encontrados:", found_files)

renamed_files = []
missing_files = []


for right_name, regex in expected_files.items():
    if right_name in found_files:
        continue

    wrong_name = None
    for i in found_files:
        if re.match(regex, i):
            wrong_name = i
            break

    if wrong_name:
        print(f"Renomeando {wrong_name} -> {right_name}")
        dbutils.fs.mv(f"{LANDING_PATH}/{wrong_name}", f"{LANDING_PATH}/{right_name}")
        found_files.remove(wrong_name)
        found_files.append(right_name)
        renamed_files.append((wrong_name, right_name))
    else:
        missing_files.append(right_name)

if missing_files:
    print(f"{FAIL} Arquivos faltando detectados")
    raise Exception(f"Arquivos faltando: {missing_files}")
else:
    print(f"{OK} Todos os arquivos estão presentes.")

### Leitura dos arquivos para inspeção

Os arquivos `chamados_hora.csv` e `clientes.csv` são lidos em `ISO-8859-1`, seguindo a mesma configuração da Bronze. Os demais utilizam UTF-8.

Mantemos todas as colunas como texto nesta etapa para inspecionar os valores antes do tratamento.

In [0]:
from functools import reduce
from pyspark.sql import functions as F

for i in expected_files:
    print(f"\n{'='*40}\n--- {i} ---")
    
    # Le como string (padrão do Bronze)
    codificacao = (
        "ISO-8859-1"
        if i in ["chamados_hora.csv", "clientes.csv"]
        else "UTF-8"
    )

    df = spark.read.csv(
        f"{LANDING_PATH}/{i}",
        header=True,
        inferSchema=False,
        encoding=codificacao
    )
    
    df.printSchema()
    
    # Mostra as primeiras linhas
    print("Amostra de dados:")
    display(df.limit(100))
    
    # Inspeção de assinatura em todas as colunas
    dfs_assinaturas = []
    
    for col_name in df.columns:
        df_sig = (
            df.select(
                F.lit(col_name).alias("coluna"),
                F.regexp_replace(
                    F.regexp_replace(
                        F.trim(F.lower(F.col(col_name))), 
                        r"\d", "9"
                    ),
                    r"[a-z]", "A"
                ).alias("assinatura")
            )
            .groupBy("coluna", "assinatura")
            .count()
        )
        dfs_assinaturas.append(df_sig)
    
    # Mostra o resultado de todas as colunas
    df_todas_assinaturas = reduce(lambda df1, df2: df1.unionByName(df2), dfs_assinaturas)
    
    print(f"Assinaturas de {i}:")
    display(
        df_todas_assinaturas.orderBy("coluna", F.col("count").desc()).limit(100)
    )